# Week 7 — Statistical Business Analysis
Introduction to Statistics for Data Science. This beginner-friendly notebook follows the same step-by-step style as the Week 5 project.

**Objectives:** descriptive statistics, distribution analysis, correlation and covariance, hypothesis testing, p-values, confidence intervals and regression.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

DATA = Path("data")
OUT = Path("outputs")
CHARTS = Path("visualizations")
OUT.mkdir(exist_ok=True)
CHARTS.mkdir(exist_ok=True)

sales = pd.read_csv(DATA / "sales_data.csv", parse_dates=["Date"])
churn = pd.read_csv(DATA / "customer_churn.csv")
sales.head()

## 1. Explore and check the datasets

In [ ]:
print("Sales shape:", sales.shape)
print("Customer churn shape:", churn.shape)
display(sales.info())
display(sales.isna().sum())
display(churn.isna().sum())
print("Duplicate sales rows:", sales.duplicated().sum())

## 2. Descriptive statistics: mean, median, mode and standard deviation

In [ ]:
columns = ["Quantity", "Price", "Total_Sales"]
summary = sales[columns].agg(["count", "mean", "median", "std", "var", "min", "max"]).T
summary["mode"] = sales[columns].mode().iloc[0]
display(summary)
print("Total sales:", sales["Total_Sales"].sum())
summary.to_csv(OUT / "descriptive_statistics.csv")

## 3. Distribution analysis and normality

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(sales["Total_Sales"], bins=15, edgecolor="black")
plt.title("Distribution of Transaction Sales")
plt.xlabel("Total Sales")
plt.ylabel("Frequency")
plt.show()

print("Skewness:", sales["Total_Sales"].skew())
print("Shapiro-Wilk test:", stats.shapiro(sales["Total_Sales"]))

## 4. Correlation and covariance

In [ ]:
corr = sales[columns].corr(method="pearson")
cov = sales[columns].cov()
display(corr)
display(cov)
corr.to_csv(OUT / "correlation_matrix.csv")
cov.to_csv(OUT / "covariance_matrix.csv")

plt.figure(figsize=(7, 5))
plt.imshow(corr.values, aspect="auto")
plt.colorbar(label="Pearson correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=25, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 5. Hypothesis testing

Significance level: **α = 0.05**.

- Null hypothesis (H₀): assumes no difference or no association.
- Alternative hypothesis (H₁): states a difference or association exists.
- If p-value < 0.05, reject H₀. Otherwise, fail to reject H₀.

### Test 1: Sales across regions (one-way ANOVA)
H₀: all regional mean sales are equal.
H₁: at least one regional mean is different.

In [ ]:
region_groups = [group["Total_Sales"].values for _, group in sales.groupby("Region")]
region_anova = stats.f_oneway(*region_groups)
print(region_anova)

### Test 2: Sales across products (one-way ANOVA)
H₀: all product mean sales are equal.
H₁: at least one product mean is different.

In [ ]:
product_groups = [group["Total_Sales"].values for _, group in sales.groupby("Product")]
product_anova = stats.f_oneway(*product_groups)
print(product_anova)

### Test 3: East vs West sales (Welch independent t-test)
H₀: East and West mean transaction sales are equal.
H₁: East and West mean transaction sales are different.

In [ ]:
east = sales.loc[sales["Region"] == "East", "Total_Sales"]
west = sales.loc[sales["Region"] == "West", "Total_Sales"]
east_west_test = stats.ttest_ind(east, west, equal_var=False)
print("East mean:", east.mean(), "| West mean:", west.mean())
print(east_west_test)

### Supplementary test: Contract type and churn (chi-square)
H₀: contract type and churn status are independent.
H₁: contract type and churn status are associated.

In [ ]:
contract_table = pd.crosstab(churn["Contract"], churn["Churn"])
chi_contract = stats.chi2_contingency(contract_table)
display(contract_table)
print(chi_contract)
contract_table.to_csv(OUT / "contract_churn_crosstab.csv")

## 6. Confidence intervals and margin of error

In [ ]:
mean_sales = sales["Total_Sales"].mean()
sales_ci = stats.t.interval(
    0.95, len(sales) - 1,
    loc=mean_sales,
    scale=stats.sem(sales["Total_Sales"])
)
margin_of_error = (sales_ci[1] - sales_ci[0]) / 2
print("Mean sales:", mean_sales)
print("95% confidence interval:", sales_ci)
print("Margin of error:", margin_of_error)

churn_ci = stats.binomtest(int(churn["Churn"].sum()), len(churn)).proportion_ci(
    confidence_level=0.95, method="wilson"
)
print("Churn rate:", churn["Churn"].mean())
print("95% Wilson interval:", churn_ci)

## 7. Regression analysis

The supplied data calculates `Total_Sales` as `Quantity × Price`. The interaction-term regression below reconstructs this formula. Its high R² is expected because the target is mathematically defined by the predictors; it is not evidence of a useful independent forecasting model.

In [ ]:
X = sales[["Quantity", "Price"]].copy()
X["Quantity_x_Price"] = X["Quantity"] * X["Price"]
y = sales["Total_Sales"]

model = LinearRegression().fit(X, y)
predictions = model.predict(X)
print("Intercept:", model.intercept_)
print("Coefficients:", dict(zip(X.columns, model.coef_)))
print("R-squared:", r2_score(y, predictions))
print("RMSE:", mean_squared_error(y, predictions) ** 0.5)

## 8. Supplementary customer churn analysis

In [ ]:
print("Overall churn rate:", churn["Churn"].mean())
churn_summary = churn.groupby("Churn").agg(
    Customers=("CustomerID", "count"),
    Average_Tenure=("Tenure", "mean"),
    Average_Monthly_Charges=("MonthlyCharges", "mean"),
    Average_Total_Charges=("TotalCharges", "mean")
)
display(churn_summary)
churn_summary.to_csv(OUT / "churn_summary.csv")

(churn.groupby("Contract")["Churn"].mean() * 100).plot(kind="bar")
plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract")
plt.ylabel("Churn Rate (%)")
plt.show()

## 9. Business interpretation and limitations

1. Use the regional and product ANOVA results to assess whether the sample provides evidence of differences in average sales.
2. Review the East–West pairwise result separately; a pairwise result does not replace the overall ANOVA.
3. Examine the contract/churn association and consider retention experiments for contract groups with higher observed churn.
4. The sales dataset has no marketing spend field, so marketing spend versus revenue cannot be tested.
5. The churn dataset is supplementary and its customer IDs are not verified to match the sales customer IDs.
6. Sales value is not profit because product costs, discounts and returns are not provided.

## 10. Export statistical test results

In [ ]:
alpha = 0.05
tests = pd.DataFrame([
    ["Sales by region (ANOVA)", region_anova.statistic, region_anova.pvalue],
    ["Sales by product (ANOVA)", product_anova.statistic, product_anova.pvalue],
    ["East vs West (Welch t-test)", east_west_test.statistic, east_west_test.pvalue],
    ["Contract vs churn (chi-square)", chi_contract.statistic, chi_contract.pvalue],
], columns=["Test", "Statistic", "P_Value"])
tests["Decision"] = np.where(tests["P_Value"] < alpha, "Reject H0", "Fail to reject H0")
display(tests)
tests.to_csv(OUT / "hypothesis_test_results.csv", index=False)